In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost', 'pyarrow', 'polars'])
import os, gc
import pickle
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import xgboost as xgb
import polars as pl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import optuna
from optuna.samplers import TPESampler

INPUT_DIR = '/kaggle/input/datasets/b22dckh072/file05/' 
TRAIN_PATH = os.path.join(INPUT_DIR, 'train_interactions.parquet')
META_PATH  = os.path.join(INPUT_DIR, 'filtered_metadata.parquet')
CAND_PATH  = os.path.join(INPUT_DIR, 'candidates_phase2.parquet')
FEAT_OUT   = os.path.join(INPUT_DIR, 'features.parquet')
TEST_PATH  = os.path.join(INPUT_DIR, 'test_interactions.parquet')

FEATURES = [
    'user_total_actions', 
    'item_total_sales', 
    'price', 
    'average_rating',
    'rating_number',
    'user_avg_rating_given',
    'item_actual_avg_rating',  
    'item_verified_ratio',      # Tỷ lệ mua thật
    'item_log_helpful_votes',   # Điểm hữu ích (Log)
    'store',                    # Tên thương hiệu (Categorical)
    'main_category',       #  Loại trang phục (Categorical)
    'sasrec_score', 
    'lightgcn_score'
]

In [ ]:
print("Đang nạp file đặc trưng bằng Polars ")
df_train = pl.read_parquet(FEAT_OUT)
X_train = df_train.select(FEATURES + ['mapped_user_id'])
y_train = df_train.select(['label'])

print("Đang chuyển qua Pandas và khóa từ điển Categorical...")
X_train_pd = X_train.to_pandas()
y_train_pd = y_train.to_pandas()
print("Đang sắp xếp dữ liệu theo User ID (Yêu cầu bắt buộc của XGBoost Ranker)...")
X_train_pd['label'] = y_train_pd['label'] # Gắn tạm label vào để sort đồng bộ
X_train_pd = X_train_pd.sort_values('mapped_user_id').reset_index(drop=True)

# Tách lại y_train_pd sau khi sort
y_train_pd = X_train_pd[['label']]
# Tách qid (Mã khách hàng)
qid_train = X_train_pd['mapped_user_id']
# Loại bỏ các cột không phải Feature ra khỏi tập X
X_train_pd = X_train_pd.drop(columns=['mapped_user_id', 'label'])

X_train_pd['store'] = X_train_pd['store'].astype(str).astype('category')
X_train_pd['main_category'] = X_train_pd['main_category'].astype(str).astype('category')

with open('/kaggle/working/category_dict.pkl', 'wb') as f:
    pickle.dump({
        'store': X_train_pd['store'].cat.categories,
        'main_category': X_train_pd['main_category'].cat.categories
    }, f)
print("Đã lưu từ điển tại /kaggle/working/category_dict.pkl")

print("Đang chuyển đổi cấu trúc dữ liệu cho XGBoost...")
dtrain = xgb.DMatrix(X_train_pd, label=y_train_pd, qid=qid_train, enable_categorical=True)

del X_train_pd, y_train_pd, df_train, X_train, y_train, qid_train
gc.collect()

constraints = {
    'sasrec_score': 1,    
    'lightgcn_score': 1   
}

# 1. ĐỊNH NGHĨA HÀM MỤC TIÊU CHO OPTUNA
def objective(trial):
    param = {
        'tree_method': 'hist',       
        'device': 'cuda',            
        'objective': 'rank:pairwise', 
        'eval_metric': 'ndcg@10', 
        
        # Các tham số Optuna
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        
        'monotone_constraints': constraints
    }

    cv_results = xgb.cv(
        param,
        dtrain,
        num_boost_round=300, 
        nfold=3,             # Chia 3 tập để test chéo 
        early_stopping_rounds=20, # Dừng sớm nếu 20 vòng không cải thiện
        metrics='ndcg@10',
        seed=42,
        verbose_eval=False
    )
    best_ndcg = cv_results['test-ndcg@10-mean'].max()
    
    # Ghi nhận số vòng lặp tối ưu cho cấu hình này
    trial.set_user_attr("best_num_trees", cv_results.shape[0])
    
    return best_ndcg

print("Bắt đầu chiến dịch dò tìm siêu tham số với Optuna (30 Trials)...")
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True) 

print("=========================================")
print(f"Điểm NDCG tốt nhất tìm được: {study.best_value:.4f}")
print("Cấu hình siêu tham số tốt nhất:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("=========================================")

print("\nTiến hành huấn luyện mô hình...")
best_params = study.best_params
best_params['tree_method'] = 'hist'
best_params['device'] = 'cuda'
best_params['objective'] = 'rank:pairwise'
best_params['eval_metric'] = 'ndcg@10'
best_params['monotone_constraints'] = constraints
best_trees = study.best_trial.user_attrs["best_num_trees"]

model = xgb.train(best_params, dtrain, num_boost_round=best_trees)

# Lưu mô hình
model_path = '/kaggle/working/xgboost_ranking_model.json'
model.save_model(model_path)
print(f"Huấn luyện hoàn tất! Đã lưu mô hình tinh anh tại: {model_path}")
xgb.plot_importance(model, importance_type='gain')
plt.show()

try:
    optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.show()
except:
    pass
del dtrain
gc.collect()

In [ ]:
import polars as pl

FEAT_OUT = '/kaggle/input/datasets/b22dckh072/file05/features.parquet' 

print("Đang quét X-quang tập dữ liệu Huấn luyện (Train Data)...")
df_train = pl.read_parquet(FEAT_OUT)
stats = df_train.group_by('label').agg([
    pl.len().alias('Tổng số dòng'),
    pl.col('sasrec_score').mean().alias('Điểm SASRec (TB)'),
    pl.col('lightgcn_score').mean().alias('Điểm LightGCN (TB)'),
    (pl.col('sasrec_score') == 0).sum().alias('Số món bị 0 điểm SASRec'),
]).sort('label', descending=True)

print(stats)

In [ ]:
import polars as pl

df_train = pl.read_parquet(FEAT_OUT)

positives = df_train.filter(pl.col('label') == 1)

print("Tổng số món khách mua thật (Nhãn 1):", positives.height)
print("Số món do SASRec đoán trúng (Score > 0):", positives.filter(pl.col('sasrec_score') > 0.0).height)
print("Số món do LightGCN đoán trúng (Score > 0):", positives.filter(pl.col('lightgcn_score') > 0.0).height)